# Tic-tac-toe minimax algorithm with alpha-beta pruning

In [1]:
import numpy as np

The original code has been extended with an implementation of the minimax algorithm with alpha-beta pruning.

The **minmax** method was changed; it now has the parameters alfa, beta

In [ ]:
class State:
    """Captures the state of a tic-tac-toe game.

    gameplan       - 3x3 numpy array
                   - 0 - empty cell
                   - 1 - X
                   - 2 - O
    player         - the player whose turn it is in the actual game
    current_player - the player making a move in the current search state
    depth          - number of moves played so far
    max_depth      - maximum search depth from the current position
    """

    generated = 0

    def __init__(self, gameplan, player, current_player=None, depth=0, max_depth=100):
        self.gameplan = gameplan
        self.player = player
        if current_player is None:
            self.current_player = player
        else:
            self.current_player = current_player
        self.depth = depth
        self.max_depth = max_depth
        State.generated += 1

    def terminal_test(self):
        """Check whether the game is over and return the winner.

        Returns:
            0 - game is not over
            1 - player 1 wins
            2 - player 2 wins
        """
        for i in range(3):
            if np.array_equal(self.gameplan[i], [1, 1, 1]):
                return 1
            if np.array_equal(self.gameplan[i], [2, 2, 2]):
                return 2
            if np.array_equal(self.gameplan[:, i], [1, 1, 1]):
                return 1
            if np.array_equal(self.gameplan[:, i], [2, 2, 2]):
                return 2

        if np.array_equal(self.gameplan.diagonal(), [1, 1, 1]):
            return 1
        if np.array_equal(self.gameplan.diagonal(), [2, 2, 2]):
            return 2
        if np.array_equal(np.fliplr(self.gameplan).diagonal(), [1, 1, 1]):
            return 1
        if np.array_equal(np.fliplr(self.gameplan).diagonal(), [2, 2, 2]):
            return 2

        return 0

    def utility(self, result):
        """Evaluate the state from the perspective of self.player.

        Returns:
             0  - draw
             1  - self.player wins
            -1  - opponent wins
        """
        if result == 0:
            return 0
        elif result == self.player:
            return 1
        else:
            return -1

    def possible_actions(self):
        """Return a list of possible actions (coordinates of empty cells)."""
        possible_actions = []
        for i in range(3):
            for j in range(3):
                if self.gameplan[i][j] == 0:
                    possible_actions.append((i, j))
        return possible_actions

    def expand(self, select_action):
        """Apply the given action and return the resulting new State.

        Switches current_player to the opponent and increments depth.
        Returns None if the action is invalid.
        """
        if select_action[0] not in range(3):
            return None
        if select_action[1] not in range(3):
            return None
        if self.gameplan[select_action[0], select_action[1]] != 0:
            return None

        new_array = np.copy(self.gameplan)
        new_array[select_action[0], select_action[1]] = self.current_player
        return State(new_array,
                     self.player,
                     self.next_current_player(),
                     self.depth + 1,
                     max_depth=self.max_depth)

    def minmax(self, strategy="max", alfa=float('-inf'), beta=float('inf'), search_depth=0):
        """Select the best action using minimax with alpha-beta pruning and depth limiting.

        strategy     - 'max' to maximise utility, 'min' to minimise it
        alfa         - lower bound (alpha cutoff for MIN nodes)
        beta         - upper bound (beta cutoff for MAX nodes)
        search_depth - levels searched so far in the current minimax call
        Returns (utility_value, action).
        """
        # terminal state — no action available, return utility directly
        result = self.terminal_test()
        if result != 0:
            return self.utility(result), None

        if strategy == "max":
            selected_utilization_value = float('-inf')
            next_strategy = "min"
        else:
            selected_utilization_value = float('inf')
            next_strategy = "max"

        actions = self.possible_actions()
        selected_action = actions[0]

        for action in actions:
            expanded_state = self.expand(action)

            result = expanded_state.terminal_test()

            if result != 0:
                return expanded_state.utility(result), action
            else:
                if len(expanded_state.possible_actions()) == 0:
                    utilization = 0
                else:
                    # !!! DOTO
                    # add condition depth limit reached — return neutral evaluation
                    utilization, _ = expanded_state.minmax(next_strategy, alfa, beta, search_depth + 1)

                if strategy == "max":
                    if utilization > selected_utilization_value:
                        selected_utilization_value = utilization
                        selected_action = action

                    # beta cutoff — opponent will never allow this
                    if selected_utilization_value >= beta:
                        return selected_utilization_value, selected_action

                    if selected_utilization_value > alfa:
                        alfa = selected_utilization_value
                else:
                    if utilization < selected_utilization_value:
                        selected_utilization_value = utilization
                        selected_action = action

                    # alpha cutoff — maximiser will never allow this
                    if selected_utilization_value <= alfa:
                        return selected_utilization_value, selected_action

                    if selected_utilization_value < beta:
                        beta = selected_utilization_value

        return selected_utilization_value, selected_action

    def next_current_player(self):
        """Return the opponent of current_player."""
        return 3 - self.current_player

    def next_player(self):
        """Return the opponent of player."""
        return 3 - self.player

## A game of tic-tac-toe

Creating the initial state of the game
* The game board is empty (0)
* It is player 1's turn

In [3]:
state = State(gameplan=np.array([[0, 0, 0], [0, 0, 0], [0, 0, 0]]),
              player=1)

The game loop remains unchanged.

Run the game and compare the game result, times, and number of generated states against the original minimax version.

In [4]:
while True:
    # check if the game is over
    game_result = state.terminal_test()
    if game_result != 0:
        print(f"Winner is {game_result} ")
        break

    # check for draw
    if len(state.possible_actions()) == 0:
        print("Drawn")
        break

    # select best action using minimax with alpha-beta pruning
    print(f"=====================\nPlayer {state.player}")
    _, player_action = state.minmax("max")
    print(f"Select action: {player_action}")
    state = state.expand(player_action)
    print(state.gameplan)
    print(f"Generated states {State.generated}.")
    State.generated = 0

    # switch to the other player
    state.player = state.next_player()

Player 1
Select action: (0, 0)
[[1 0 0]
 [0 0 0]
 [0 0 0]]
Generated states 17630.
Player 2
Select action: (1, 1)
[[1 0 0]
 [0 2 0]
 [0 0 0]]
Generated states 2124.
Player 1
Select action: (0, 1)
[[1 1 0]
 [0 2 0]
 [0 0 0]]
Generated states 743.
Player 2
Select action: (0, 2)
[[1 1 2]
 [0 2 0]
 [0 0 0]]
Generated states 61.
Player 1
Select action: (2, 0)
[[1 1 2]
 [0 2 0]
 [1 0 0]]
Generated states 50.
Player 2
Select action: (1, 0)
[[1 1 2]
 [2 2 0]
 [1 0 0]]
Generated states 17.
Player 1
Select action: (1, 2)
[[1 1 2]
 [2 2 1]
 [1 0 0]]
Generated states 10.
Player 2
Select action: (2, 1)
[[1 1 2]
 [2 2 1]
 [1 2 0]]
Generated states 5.
Player 1
Select action: (2, 2)
[[1 1 2]
 [2 2 1]
 [1 2 1]]
Generated states 2.
Drawn


# Task
Add a limit on the maximum search depth to the algorithm.

Again, you need to change the code at the place marked # **!!! todo**

In [5]:
state = State(gameplan=np.array([[0, 0, 0], [0, 0, 0], [0, 0, 0]]),
              player=1, max_depth=2)

In [6]:
while True:
    # check if the game is over
    game_result = state.terminal_test()
    if game_result != 0:
        print(f"Winner is {game_result} ")
        break

    # check for draw
    if len(state.possible_actions()) == 0:
        print("Drawn")
        break

    # select best action using minimax with alpha-beta pruning and depth limit
    print(f"=====================\nPlayer {state.player}")
    _, player_action = state.minmax("max")
    print(f"Select action: {player_action}")
    state = state.expand(player_action)
    print(state.gameplan)
    print(f"Generated states {State.generated}.")
    State.generated = 0

    # switch to the other player
    state.player = state.next_player()

Player 1
Select action: (0, 0)
[[1 0 0]
 [0 0 0]
 [0 0 0]]
Generated states 27.
Player 2
Select action: (0, 1)
[[1 2 0]
 [0 0 0]
 [0 0 0]]
Generated states 23.
Player 1
Select action: (0, 2)
[[1 2 1]
 [0 0 0]
 [0 0 0]]
Generated states 20.
Player 2
Select action: (1, 0)
[[1 2 1]
 [2 0 0]
 [0 0 0]]
Generated states 17.
Player 1
Select action: (1, 1)
[[1 2 1]
 [2 1 0]
 [0 0 0]]
Generated states 14.
Player 2
Select action: (1, 2)
[[1 2 1]
 [2 1 2]
 [0 0 0]]
Generated states 13.
Player 1
Select action: (2, 0)
[[1 2 1]
 [2 1 2]
 [1 0 0]]
Generated states 2.
Winner is 1 
